In [1]:
import pandas as pd
import numpy as np
import os
import logging
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [2]:
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted successfully!")
print("✓ Your files are now available at /content/drive/My Drive/")

Mounted at /content/drive
✓ Google Drive mounted successfully!
✓ Your files are now available at /content/drive/My Drive/


In [3]:
import os
import sys
from pathlib import Path

# Define base paths for Colab
COLAB_PROJECT_ROOT = Path("/content/drive/My Drive/godavari_gnn_project")

In [4]:
import pandas as pd
import numpy as np
import os
import logging
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

def load_nwis_data(file_path, data_type):
    """Load NWIS Excel files with proper header handling and date parsing"""
    try:
        # Read the Excel file without headers first to find data start
        df_raw = pd.read_excel(file_path, header=None)

        # Find the row where actual data starts (look for 'Data Type Code' header)
        data_start_row = None
        for idx, row in df_raw.iterrows():
            # Convert all row values to string for a robust search, as values can be mixed types
            if 'Data Type Code' in [str(x) for x in row.values]:
                data_start_row = idx
                break

        if data_start_row is None:
            print(f"⚠️  Could not find 'Data Type Code' header in {file_path.name}. Trying alternative patterns.")
            # Try to find alternative header patterns if primary is not found
            for idx, row in df_raw.iterrows():
                if any(col in [str(x) for x in row.values] for col in ['Data Time', 'Data Value', 'Unit']):
                    data_start_row = idx
                    break

        if data_start_row is not None:
            # Read data with proper header
            df = pd.read_excel(file_path, header=data_start_row)
            print(f"✅ Data starts at row {data_start_row + 1} in {file_path.name}")

            # Ensure 'Data Time' column is in datetime format
            if 'Data Time' in df.columns:
                # Coerce errors will turn unparseable dates into NaT (Not a Time)
                df['Data Time'] = pd.to_datetime(df['Data Time'], errors='coerce')
                # Drop rows where 'Data Time' could not be parsed (is NaT)
                df.dropna(subset=['Data Time'], inplace=True)

                if df['Data Time'].empty and not df.empty:
                    print(f"❌ No valid 'Data Time' entries left after parsing and cleaning for {file_path.name}.")
                    return None
                elif df.empty:
                    print(f"❌ DataFrame became empty after cleaning 'Data Time' for {file_path.name}.")
                    return None
            else:
                print(f"⚠️  'Data Time' column not found in {file_path.name} after header detection. Data might be incomplete.")
                # If 'Data Time' is critical and missing, you might want to return None here
                # For now, we'll proceed but this might lead to errors downstream if 'Data Time' is expected.

            return df
        else:
            print(f"❌ No data header found in {file_path.name}. Cannot load data.")
            return None

    except Exception as e:
        print(f"❌ Error reading {file_path.name}: {e}")
        return None

def load_lake_data(lake_name):
    """Load all data files for a specific lake with NWIS format handling"""
    lake_path = COLAB_PROJECT_ROOT / 'data' / 'raw' / lake_name.lower()

    data_dict = {}

    try:
        # Define exact file names
        file_patterns = {
            'rainfall': f'Rainfall_{lake_name.capitalize()}.xlsx',
            'humidity': f'Relative-Humidity_{lake_name.capitalize()}.xlsx',
            'water_level': f'River-Water-Level_{lake_name.capitalize()}.xlsx',
            'discharge': f'River-Water-Discharge_{lake_name.capitalize()}.xlsx'
        }

        # Load each file
        for data_type, filename in file_patterns.items():
            file_path = lake_path / filename
            if file_path.exists():
                df = load_nwis_data(file_path, data_type) # Call load_nwis_data here
                if df is not None:
                    data_dict[data_type] = df
                    print(f"✅ Loaded {data_type} data for {lake_name}")
                else:
                    print(f"⚠️  Could not process {data_type} data for {lake_name}")
            else:
                print(f"⚠️  File not found: {file_path}")

    except Exception as e:
        print(f"❌ Error loading data for {lake_name}: {e}")

    return data_dict

# Test with updated loader
print("Loading Adhala data with NWIS format handling...")
adhala_data = load_lake_data('adhala')
print(f"Adhala data keys: {list(adhala_data.keys())}")

# Check the structure after proper loading
if 'rainfall' in adhala_data and adhala_data['rainfall'] is not None:
    print("\n📊 Rainfall data structure after proper loading:")
    print(adhala_data['rainfall'].head())
    print(f"Columns: {list(adhala_data['rainfall'].columns)}")
    print(f"Shape: {adhala_data['rainfall'].shape}")
else:
    print("\n⚠️ Rainfall data not loaded or is empty.")

Loading Adhala data with NWIS format handling...
✅ Data starts at row 7 in Rainfall_Adhala.xlsx
✅ Loaded rainfall data for adhala
✅ Data starts at row 7 in Relative-Humidity_Adhala.xlsx
✅ Loaded humidity data for adhala
✅ Data starts at row 7 in River-Water-Level_Adhala.xlsx
✅ Loaded water_level data for adhala
✅ Data starts at row 7 in River-Water-Discharge_Adhala.xlsx
✅ Loaded discharge data for adhala
Adhala data keys: ['rainfall', 'humidity', 'water_level', 'discharge']

📊 Rainfall data structure after proper loading:
  Data Type Code                       Data Type Description  \
0            MPS  MANUAL-Rainfall - SRG(Standard Rain Gauge)   
1            MPS  MANUAL-Rainfall - SRG(Standard Rain Gauge)   
2            MPS  MANUAL-Rainfall - SRG(Standard Rain Gauge)   
3            MPS  MANUAL-Rainfall - SRG(Standard Rain Gauge)   
4            MPS  MANUAL-Rainfall - SRG(Standard Rain Gauge)   

            Data Time  Data Value Unit  
0 2021-06-01 08:30:00         0.0   mm  
1 202

In [5]:
def process_all_lakes():
    """Process all 6 lakes in one go"""
    lakes = ['adhala', 'girija', 'indravati', 'manjira', 'valamuru', 'sabari']

    all_lakes_data = {}

    print("🚀 STARTING BATCH PROCESSING FOR ALL 6 LAKES")
    print("=" * 60)

    for lake_name in lakes:
        print(f"\n{'='*50}")
        print(f"🌊 PROCESSING: {lake_name.upper()}")
        print(f"{'='*50}")

        # Load data for this lake
        lake_data = load_lake_data(lake_name)

        if lake_data:
            all_lakes_data[lake_name] = lake_data
            print(f"✅ SUCCESS: Loaded {len(lake_data)} data types for {lake_name}")

            # Show data structure for each data type
            for data_type, df in lake_data.items():
                print(f"   📊 {data_type}: {df.shape[0]} rows, {df.shape[1]} columns")
                print(f"   📅 Date range: {df['Data Time'].min()} to {df['Data Time'].max()}")
        else:
            print(f"❌ FAILED: No data loaded for {lake_name}")

    # Final summary
    print(f"\n{'='*60}")
    print("📊 BATCH PROCESSING SUMMARY")
    print(f"{'='*60}")
    print(f"✅ Successfully processed: {len(all_lakes_data)} out of {len(lakes)} lakes")

    successful_lakes = list(all_lakes_data.keys())
    failed_lakes = [lake for lake in lakes if lake not in successful_lakes]

    if successful_lakes:
        print(f"🎯 Successful lakes: {', '.join(successful_lakes)}")
    if failed_lakes:
        print(f"⚠️  Failed lakes: {', '.join(failed_lakes)}")

    # Show detailed breakdown
    print(f"\n📈 DETAILED BREAKDOWN:")
    for lake_name, lake_data in all_lakes_data.items():
        data_types = list(lake_data.keys())
        total_rows = sum([df.shape[0] for df in lake_data.values()])
        print(f"   🌊 {lake_name}: {len(data_types)} data types, {total_rows} total records")
        for data_type, df in lake_data.items():
            date_range = f"{df['Data Time'].min().strftime('%Y-%m-%d')} to {df['Data Time'].max().strftime('%Y-%m-%d')}"
            print(f"      • {data_type}: {df.shape[0]} records ({date_range})")

    return all_lakes_data

# Run the batch processing
print("Starting batch processing for all lakes...")
all_lakes_raw_data = process_all_lakes()


Starting batch processing for all lakes...
🚀 STARTING BATCH PROCESSING FOR ALL 6 LAKES

🌊 PROCESSING: ADHALA
✅ Data starts at row 7 in Rainfall_Adhala.xlsx
✅ Loaded rainfall data for adhala
✅ Data starts at row 7 in Relative-Humidity_Adhala.xlsx
✅ Loaded humidity data for adhala
✅ Data starts at row 7 in River-Water-Level_Adhala.xlsx
✅ Loaded water_level data for adhala
✅ Data starts at row 7 in River-Water-Discharge_Adhala.xlsx
✅ Loaded discharge data for adhala
✅ SUCCESS: Loaded 4 data types for adhala
   📊 rainfall: 306 rows, 5 columns
   📅 Date range: 2021-06-01 08:30:00 to 2022-10-31 08:30:00
   📊 humidity: 1340 rows, 5 columns
   📅 Date range: 2020-01-01 08:30:00 to 2021-10-31 17:30:00
   📊 water_level: 1135 rows, 5 columns
   📅 Date range: 2020-01-01 08:30:00 to 2021-10-31 17:30:00
   📊 discharge: 608 rows, 5 columns
   📅 Date range: 2020-06-01 08:30:00 to 2021-10-31 17:30:00

🌊 PROCESSING: GIRIJA
✅ Data starts at row 7 in Rainfall_Girija.xlsx
✅ Loaded rainfall data for girija
✅

#Cleaning and feature engineering

In [6]:
def clean_and_merge_lake_data(all_lakes_raw_data):
    """Clean the raw data and create unified time-series features"""
    all_lakes_processed = {}

    for lake_name, lake_data in all_lakes_raw_data.items():
        print(f"\n🔄 Processing {lake_name.upper()}...")

        processed_dfs = []

        for data_type, df in lake_data.items():
            print(f"   Cleaning {data_type}...")

            # Clean and standardize
            df_clean = clean_nwis_data(df, data_type, lake_name)
            if df_clean is not None:
                processed_dfs.append(df_clean)

        # Merge all parameters for this lake
        if processed_dfs:
            merged_df = processed_dfs[0]
            for df in processed_dfs[1:]:
                merged_df = merged_df.join(df, how='outer')

            # Add temporal features
            merged_df = add_temporal_features(merged_df, lake_name)
            all_lakes_processed[lake_name] = merged_df

            print(f"✅ {lake_name}: {len(merged_df)} days, {len(merged_df.columns)} features")
        else:
            print(f"❌ {lake_name}: No valid data after cleaning")

    return all_lakes_processed

def clean_nwis_data(df, data_type, lake_name):
    """Clean individual NWIS dataset"""
    try:
        # Keep only essential columns
        if 'Data Time' not in df.columns or 'Data Value' not in df.columns:
            print(f"⚠️  {lake_name} {data_type}: Missing required columns")
            return None

        df_clean = df[['Data Time', 'Data Value']].copy()
        df_clean.columns = ['timestamp', 'value']

        # Convert to datetime and numeric
        df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'])
        df_clean['value'] = pd.to_numeric(df_clean['value'], errors='coerce')

        # Remove invalid data
        df_clean = df_clean.dropna()

        # Set index and resample to daily
        df_clean = df_clean.set_index('timestamp')
        df_daily = df_clean.resample('D').mean()

        # Create column name based on data type
        col_name = {
            'rainfall': 'rainfall_mm',
            'humidity': 'humidity_pct',
            'water_level': 'water_level_m',
            'discharge': 'discharge_m3s'
        }.get(data_type, data_type)

        df_daily.columns = [col_name]

        return df_daily

    except Exception as e:
        print(f"❌ Error cleaning {lake_name} {data_type}: {e}")
        return None

def add_temporal_features(df, lake_name):
    """Add temporal and derived features"""
    df_enhanced = df.copy()

    # Basic temporal features
    df_enhanced['day_of_year'] = df_enhanced.index.dayofyear
    df_enhanced['month'] = df_enhanced.index.month
    df_enhanced['year'] = df_enhanced.index.year

    # Seasonal encoding
    df_enhanced['day_sin'] = np.sin(2 * np.pi * df_enhanced['day_of_year'] / 365)
    df_enhanced['day_cos'] = np.cos(2 * np.pi * df_enhanced['day_of_year'] / 365)

    # Monsoon indicator (June-Sept)
    df_enhanced['is_monsoon'] = ((df_enhanced.index.month >= 6) &
                                (df_enhanced.index.month <= 9)).astype(int)

    # Rate of change features
    if 'water_level_m' in df_enhanced.columns:
        df_enhanced['water_level_change'] = df_enhanced['water_level_m'].diff()
        df_enhanced['water_level_change_3d'] = df_enhanced['water_level_m'].diff(3)

    if 'rainfall_mm' in df_enhanced.columns:
        df_enhanced['rainfall_3d_sum'] = df_enhanced['rainfall_mm'].rolling(3).sum()
        df_enhanced['rainfall_7d_sum'] = df_enhanced['rainfall_mm'].rolling(7).sum()

    if 'discharge_m3s' in df_enhanced.columns:
        df_enhanced['discharge_change'] = df_enhanced['discharge_m3s'].diff()

    print(f"   Added {len(df_enhanced.columns) - len(df.columns)} temporal features")
    return df_enhanced

# Process the data
print("🔄 Cleaning and feature engineering...")
all_lakes_processed = clean_and_merge_lake_data(all_lakes_raw_data)

🔄 Cleaning and feature engineering...

🔄 Processing ADHALA...
   Cleaning rainfall...
   Cleaning humidity...
   Cleaning water_level...
   Cleaning discharge...
   Added 11 temporal features
✅ adhala: 1035 days, 15 features

🔄 Processing GIRIJA...
   Cleaning rainfall...
   Cleaning humidity...
   Cleaning water_level...
   Cleaning discharge...
   Added 11 temporal features
✅ girija: 732 days, 15 features

🔄 Processing INDRAVATI...
   Cleaning rainfall...
   Cleaning humidity...
   Cleaning water_level...
   Cleaning discharge...
   Added 11 temporal features
✅ indravati: 630 days, 15 features

🔄 Processing MANJIRA...
   Cleaning rainfall...
   Cleaning humidity...
   Cleaning water_level...
   Cleaning discharge...
   Added 11 temporal features
✅ manjira: 732 days, 15 features

🔄 Processing VALAMURU...
   Cleaning rainfall...
   Cleaning humidity...
   Cleaning water_level...
   Cleaning discharge...
   Added 11 temporal features
✅ valamuru: 732 days, 15 features

🔄 Processing SABAR

#Step 2: Handle Missing Data & Normalization

In [7]:
def handle_missing_data(all_lakes_processed):
    """Intelligently handle missing values"""
    from sklearn.preprocessing import StandardScaler

    all_lakes_cleaned = {}
    scalers = {}

    for lake_name, df in all_lakes_processed.items():
        df_clean = df.copy()

        print(f"\n🧹 Cleaning {lake_name}...")

        # Handle missing values with different strategies
        for column in df_clean.columns:
            if df_clean[column].isnull().any():
                missing_count = df_clean[column].isnull().sum()

                if 'water_level' in column or 'discharge' in column:
                    df_clean[column] = df_clean[column].fillna(method='ffill').interpolate()
                elif 'rainfall' in column:
                    df_clean[column] = df_clean[column].fillna(0).interpolate()
                else:
                    df_clean[column] = df_clean[column].interpolate()

                print(f"   ✅ {column}: Fixed {missing_count} missing values")

        # Final cleanup
        df_clean = df_clean.fillna(method='ffill').fillna(method='bfill')

        # Normalize features (except temporal ones)
        non_temporal_cols = [col for col in df_clean.columns if col not in
                           ['day_of_year', 'month', 'year', 'day_sin', 'day_cos', 'is_monsoon']]

        scaler = StandardScaler()
        df_clean[non_temporal_cols] = scaler.fit_transform(df_clean[non_temporal_cols])
        scalers[lake_name] = scaler

        all_lakes_cleaned[lake_name] = df_clean

    return all_lakes_cleaned, scalers

# Clean and normalize
print("\n🧹 Handling missing values and normalization...")
all_lakes_cleaned, scalers = handle_missing_data(all_lakes_processed)


🧹 Handling missing values and normalization...

🧹 Cleaning adhala...
   ✅ rainfall_mm: Fixed 729 missing values
   ✅ humidity_pct: Fixed 365 missing values
   ✅ water_level_m: Fixed 457 missing values
   ✅ discharge_m3s: Fixed 729 missing values
   ✅ water_level_change: Fixed 459 missing values
   ✅ water_level_change_3d: Fixed 463 missing values
   ✅ rainfall_3d_sum: Fixed 733 missing values
   ✅ rainfall_7d_sum: Fixed 741 missing values
   ✅ discharge_change: Fixed 731 missing values

🧹 Cleaning girija...
   ✅ rainfall_mm: Fixed 585 missing values
   ✅ discharge_m3s: Fixed 49 missing values
   ✅ water_level_change: Fixed 1 missing values
   ✅ water_level_change_3d: Fixed 3 missing values
   ✅ rainfall_3d_sum: Fixed 589 missing values
   ✅ rainfall_7d_sum: Fixed 597 missing values
   ✅ discharge_change: Fixed 72 missing values

🧹 Cleaning indravati...
   ✅ rainfall_mm: Fixed 620 missing values
   ✅ humidity_pct: Fixed 83 missing values
   ✅ water_level_m: Fixed 43 missing values
   ✅

#Step 3: Graph Construction

In [8]:
import torch

def build_complete_lake_graph():
    """Build complete graph structure for all 6 lakes in Godavari Basin"""

    # Node mapping - all 6 lakes with their positions
    node_mapping = {
        0: 'adhala',      # Upper Basin
        1: 'girija',      # Upper Basin
        2: 'indravati',   # Middle Basin
        3: 'manjira',     # Middle Basin
        4: 'valamuru',    # Lower Basin
        5: 'sabari'       # Lower Basin
    }

    # Basin grouping
    basin_groups = {
        'upper_basin': [0, 1],      # Adhala, Girija
        'middle_basin': [2, 3],     # Indravati, Manjira
        'lower_basin': [4, 5]       # Valamuru, Sabari
    }

    # Edges based on Godavari Basin hydrology
    edges = [
        # UPPER BASIN CONNECTIONS
        (0, 1), (1, 0),  # Adhala ↔ Girija (bidirectional, same sub-basin)

        # UPPER → MIDDLE BASIN CONNECTIONS
        (0, 2), (1, 2),  # Adhala → Indravati, Girija → Indravati
        (0, 3), (1, 3),  # Adhala → Manjira, Girija → Manjira

        # MIDDLE BASIN CONNECTIONS
        (2, 3), (3, 2),  # Indravati ↔ Manjira (bidirectional)

        # MIDDLE → LOWER BASIN CONNECTIONS
        (2, 4), (2, 5),  # Indravati → Valamuru, Indravati → Sabari
        (3, 4), (3, 5),  # Manjira → Valamuru, Manjira → Sabari

        # LOWER BASIN CONNECTIONS
        (4, 5), (5, 4),  # Valamuru ↔ Sabari (bidirectional, converge near end)

        # DOWNSTREAM FLOW (main river direction)
        (0, 2), (1, 2),  # Upper → Middle
        (2, 4), (3, 4),  # Middle → Lower
        (4, 5)           # Final convergence
    ]

    # Remove duplicate edges while preserving order
    edges = list(dict.fromkeys(edges))

    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

    print("🌊 COMPLETE GRAPH STRUCTURE FOR GODAVARI BASIN")
    print("=" * 50)

    print(f"📊 Graph Summary:")
    print(f"   • Total Nodes: {len(node_mapping)}")
    print(f"   • Total Edges: {len(edges)}")
    print(f"   • Density: {len(edges) / (len(node_mapping) * (len(node_mapping) - 1)):.2%}")

    print(f"\n🏞️  Basin Structure:")
    for basin_name, nodes in basin_groups.items():
        lake_names = [node_mapping[node_id] for node_id in nodes]
        print(f"   • {basin_name.replace('_', ' ').title()}: {', '.join(lake_names)}")

    print(f"\n🔗 Hydrological Connections:")
    print(f"   • Upper → Middle: Water flows from upper to middle basin")
    print(f"   • Middle → Lower: Water flows from middle to lower basin")
    print(f"   • Within Basins: Lakes in same basin influence each other")
    print(f"   • Lower Convergence: Valamuru and Sabari converge near basin end")

    print(f"\n📈 Node Degrees:")
    from collections import defaultdict
    node_degrees = defaultdict(int)
    for src, dst in edges:
        node_degrees[src] += 1

    for node_id, degree in sorted(node_degrees.items()):
        print(f"   • {node_mapping[node_id].title()}: {degree} connections")

    return edge_index, node_mapping, basin_groups

# Build the complete graph
print("🏗️ Building complete graph for all 6 lakes...")
edge_index, node_mapping, basin_groups = build_complete_lake_graph()

🏗️ Building complete graph for all 6 lakes...
🌊 COMPLETE GRAPH STRUCTURE FOR GODAVARI BASIN
📊 Graph Summary:
   • Total Nodes: 6
   • Total Edges: 14
   • Density: 46.67%

🏞️  Basin Structure:
   • Upper Basin: adhala, girija
   • Middle Basin: indravati, manjira
   • Lower Basin: valamuru, sabari

🔗 Hydrological Connections:
   • Upper → Middle: Water flows from upper to middle basin
   • Middle → Lower: Water flows from middle to lower basin
   • Within Basins: Lakes in same basin influence each other
   • Lower Convergence: Valamuru and Sabari converge near basin end

📈 Node Degrees:
   • Adhala: 3 connections
   • Girija: 3 connections
   • Indravati: 3 connections
   • Manjira: 3 connections
   • Valamuru: 1 connections
   • Sabari: 1 connections


#Step 4: Create GNN Dataset

In [9]:
def create_gnn_dataset(all_lakes_cleaned, sequence_length=30, prediction_horizon=7):
    """Create sequences for GNN training"""
    import torch
    from torch.utils.data import Dataset

    class LakeDataset(Dataset):
        def __init__(self, sequences, targets):
            self.sequences = sequences
            self.targets = targets

        def __len__(self):
            return len(self.sequences)

        def __getitem__(self, idx):
            return self.sequences[idx], self.targets[idx]

    all_sequences_stacked = []
    all_targets_stacked = []

    # Find common date range across all lakes
    common_dates = None
    # Ensure lakes are processed in a consistent order (matching node_mapping)
    sorted_lake_names = [name for id, name in sorted(node_mapping.items())]

    dfs_aligned = {}
    for lake_name in sorted_lake_names:
        df = all_lakes_cleaned[lake_name]
        if common_dates is None:
            common_dates = df.index
        else:
            common_dates = common_dates.intersection(df.index)
        dfs_aligned[lake_name] = df

    print(f"📅 Common date range: {len(common_dates)} days")

    # Get feature columns (exclude temporal encoding for model input)
    # Assuming feature_cols is already defined globally from a previous step
    if 'feature_cols' not in globals():
        # If not, determine it from one of the DFs (e.g., first lake)
        sample_df = list(all_lakes_cleaned.values())[0]
        global feature_cols # Declare global if it's meant to be used elsewhere
        feature_cols = [col for col in sample_df.columns if col not in
                       ['day_of_year', 'month', 'year', 'day_sin', 'day_cos', 'is_monsoon']]

    # Create a list of daily feature matrices (num_nodes, num_features)
    daily_feature_matrices = []
    daily_water_level_targets = []

    for date in common_dates:
        current_day_features = []
        current_day_water_levels = []
        for lake_name in sorted_lake_names:
            df_lake = dfs_aligned[lake_name]
            if date in df_lake.index:
                current_day_features.append(df_lake.loc[date, feature_cols].values)
                current_day_water_levels.append(df_lake.loc[date, 'water_level_m'])
            else:
                # Handle cases where a specific lake might not have data for a common date (should be rare after cleaning)
                # For now, let's append NaNs or zeros, then interpolate/fill before GNN input
                current_day_features.append(np.full(len(feature_cols), np.nan))
                current_day_water_levels.append(np.nan)

        daily_feature_matrices.append(np.array(current_day_features)) # Shape: (num_nodes, num_features)
        daily_water_level_targets.append(np.array(current_day_water_levels)) # Shape: (num_nodes,)

    # Convert to numpy arrays
    daily_feature_matrices = np.array(daily_feature_matrices) # Shape: (num_days, num_nodes, num_features)
    daily_water_level_targets = np.array(daily_water_level_targets) # Shape: (num_days, num_nodes)

    # Create sequences from the daily feature matrices
    num_days_available = daily_feature_matrices.shape[0]
    num_nodes = len(sorted_lake_names)

    for i in range(num_days_available - sequence_length - prediction_horizon):
        seq = daily_feature_matrices[i : i + sequence_length] # Shape: (seq_len, num_nodes, num_features)
        # Target: water_level_m for all nodes for the next prediction_horizon days
        target = daily_water_level_targets[i + sequence_length : i + sequence_length + prediction_horizon]
        target = target.T # Transpose to (num_nodes, pred_len) for consistency with model output

        all_sequences_stacked.append(seq)
        all_targets_stacked.append(target)

    # Convert to tensors
    sequences_tensor = torch.FloatTensor(np.array(all_sequences_stacked))
    targets_tensor = torch.FloatTensor(np.array(all_targets_stacked))

    print(f"\n📊 Dataset Summary:")
    print(f"   Total sequences: {len(all_sequences_stacked)}")
    print(f"   Sequence shape: {sequences_tensor.shape}")
    print(f"   Target shape: {targets_tensor.shape}")

    return LakeDataset(sequences_tensor, targets_tensor), feature_cols

# Create dataset
print("\n📦 Creating GNN dataset...")
dataset, feature_cols = create_gnn_dataset(all_lakes_cleaned)
print(f"   Features used: {feature_cols}")


📦 Creating GNN dataset...
📅 Common date range: 630 days

📊 Dataset Summary:
   Total sequences: 593
   Sequence shape: torch.Size([593, 30, 6, 9])
   Target shape: torch.Size([593, 6, 7])
   Features used: ['rainfall_mm', 'humidity_pct', 'water_level_m', 'discharge_m3s', 'water_level_change', 'water_level_change_3d', 'rainfall_3d_sum', 'rainfall_7d_sum', 'discharge_change']


#Step 5: Save Processed Data

In [10]:
def save_processed_data(all_lakes_cleaned, dataset, edge_index, node_mapping):
    """Save all processed data for GNN training"""
    processed_path = COLAB_PROJECT_ROOT / 'data' / 'processed'
    processed_path.mkdir(parents=True, exist_ok=True)

    # Save individual lake data
    for lake_name, df in all_lakes_cleaned.items():
        csv_path = processed_path / f'{lake_name}_cleaned.csv'
        df.to_csv(csv_path)
        print(f"💾 Saved {lake_name} to {csv_path}")

    # Save graph structure
    graph_data = {
        'edge_index': edge_index,
        'node_mapping': node_mapping,
        'feature_columns': feature_cols
    }

    graph_path = processed_path / 'graph_data.pkl'
    with open(graph_path, 'wb') as f:
        pickle.dump(graph_data, f)
    print(f"💾 Saved graph data to {graph_path}")

    # Save dataset info
    dataset_info = {
        'sequence_length': 30,
        'prediction_horizon': 7,
        'total_sequences': len(dataset),
        'feature_dim': dataset.sequences.shape[-1]
    }

    info_path = processed_path / 'dataset_info.json'
    with open(info_path, 'w') as f:
        json.dump(dataset_info, f, indent=2)
    print(f"💾 Saved dataset info to {info_path}")

    print(f"\n🎉 All processed data saved!")

# Save everything
import pickle
import json

save_processed_data(all_lakes_cleaned, dataset, edge_index, node_mapping)

💾 Saved adhala to /content/drive/My Drive/godavari_gnn_project/data/processed/adhala_cleaned.csv
💾 Saved girija to /content/drive/My Drive/godavari_gnn_project/data/processed/girija_cleaned.csv
💾 Saved indravati to /content/drive/My Drive/godavari_gnn_project/data/processed/indravati_cleaned.csv
💾 Saved manjira to /content/drive/My Drive/godavari_gnn_project/data/processed/manjira_cleaned.csv
💾 Saved valamuru to /content/drive/My Drive/godavari_gnn_project/data/processed/valamuru_cleaned.csv
💾 Saved sabari to /content/drive/My Drive/godavari_gnn_project/data/processed/sabari_cleaned.csv
💾 Saved graph data to /content/drive/My Drive/godavari_gnn_project/data/processed/graph_data.pkl
💾 Saved dataset info to /content/drive/My Drive/godavari_gnn_project/data/processed/dataset_info.json

🎉 All processed data saved!


#Step 6: Quick Data Analysis

In [11]:
def analyze_processed_data(all_lakes_cleaned):
    """Analyze the final processed data"""
    print("\n📊 FINAL PROCESSED DATA ANALYSIS")
    print("=" * 50)

    for lake_name, df in all_lakes_cleaned.items():
        print(f"\n🌊 {lake_name.upper()}:")
        print(f"   📅 Date range: {df.index.min()} to {df.index.max()}")
        print(f"   📈 Total days: {len(df)}")
        print(f"   🔧 Features: {len(df.columns)}")

        # Show key statistics
        key_features = ['water_level_m', 'rainfall_mm', 'discharge_m3s', 'humidity_pct']
        for feature in key_features:
            if feature in df.columns:
                stats = df[feature].describe()
                print(f"   📊 {feature}: mean={stats['mean']:.2f}, std={stats['std']:.2f}")

# Analyze final data
analyze_processed_data(all_lakes_cleaned)


📊 FINAL PROCESSED DATA ANALYSIS

🌊 ADHALA:
   📅 Date range: 2020-01-01 00:00:00 to 2022-10-31 00:00:00
   📈 Total days: 1035
   🔧 Features: 15
   📊 water_level_m: mean=0.00, std=1.00
   📊 rainfall_mm: mean=0.00, std=1.00
   📊 discharge_m3s: mean=0.00, std=1.00
   📊 humidity_pct: mean=-0.00, std=1.00

🌊 GIRIJA:
   📅 Date range: 2020-01-01 00:00:00 to 2022-01-01 00:00:00
   📈 Total days: 732
   🔧 Features: 15
   📊 water_level_m: mean=-0.00, std=1.00
   📊 rainfall_mm: mean=0.00, std=1.00
   📊 discharge_m3s: mean=0.00, std=1.00
   📊 humidity_pct: mean=-0.00, std=1.00

🌊 INDRAVATI:
   📅 Date range: 2020-01-01 00:00:00 to 2021-09-21 00:00:00
   📈 Total days: 630
   🔧 Features: 15
   📊 water_level_m: mean=0.00, std=1.00
   📊 rainfall_mm: mean=0.00, std=0.00
   📊 discharge_m3s: mean=-0.00, std=1.00
   📊 humidity_pct: mean=0.00, std=1.00

🌊 MANJIRA:
   📅 Date range: 2020-01-01 00:00:00 to 2022-01-01 00:00:00
   📈 Total days: 732
   🔧 Features: 15
   📊 water_level_m: mean=-0.00, std=1.00
   📊 r

#GNN model building

In [12]:
import torch

def get_torch_version():
    return torch.__version__

def get_cuda_version():
    if torch.cuda.is_available():
        return torch.version.cuda
    return "CUDA not available"

print(f"PyTorch Version: {get_torch_version()}")
print(f"CUDA Version: {get_cuda_version()}")


PyTorch Version: 2.9.0+cu128
CUDA Version: 12.8


In [ ]:
# Install torch-geometric and its dependencies
import torch

# Determine the CUDA version based on the PyTorch installation
# If CUDA is available, use the CUDA-specific installation; otherwise, use CPU-only

if torch.cuda.is_available():
    # Example for CUDA 11.8. You might need to adjust based on your specific CUDA version.
    # Check https://pytorch-geometric.readthedocs.io/en/latest/install/installation.html for the correct command.
    # A common approach is to install based on PyTorch and CUDA versions.
    # For Colab, often `cu118` is relevant for PyTorch 2.x and CUDA 11.8.
    # Let's get the specific PyTorch version for dynamic installation if possible.
    TORCH_VERSION = torch.__version__
    # Example: '2.1.0+cu118' -> 'cu118'
    CUDA_VERSION = 'cpu' # Default to cpu, then try to extract
    if '+cu' in TORCH_VERSION:
        CUDA_VERSION = 'cu' + TORCH_VERSION.split('+cu')[1].split('.')[0]
    elif 'cuda' in TORCH_VERSION:
        CUDA_VERSION = 'cu' + TORCH_VERSION.split('cuda')[1].replace('.', '')

    # Simplified for common Colab setups (assuming recent PyTorch & CUDA 11.8)
    # This often works for PyTorch 2.x and above on Colab
    print(f"Installing for PyTorch {TORCH_VERSION} with CUDA {CUDA_VERSION}")
    !pip install torch_geometric torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-{TORCH_VERSION.split('+')[0]}%2B{CUDA_VERSION}.html

else:
    print("CUDA not available. Installing CPU-only version of torch_geometric.")
    !pip install torch_geometric


Installing for PyTorch 2.9.0+cu128 with CUDA cu128
Looking in links: https://data.pyg.org/whl/torch-2.9.0%2Bcu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 12.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.0/210.0 kB 20.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 6.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 74.2 MB/s eta 0:00:00
  Created wheel for torch_scatter: filename=torch_scatter-2.1.2-cp312-cp312-linux_x86_64.whl size=3848821 sha256=0727680301f2621c23a0e2150764833951b12123ca55201db1a02928613da5bb
  Stored in directory: /root/.cache/pip/wheels/84/20/50/44800723f57cd798630e77b3ec83bc80bd26a1e3dc3a672ef5


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import numpy as np

class SpatioTemporalGNN(nn.Module):
    """GNN model for spatio-temporal water level prediction"""

    def __init__(self, num_features, hidden_dim=64, num_layers=2,
                 seq_len=30, pred_len=7, dropout=0.2):
        super(SpatioTemporalGNN, self).__init__()

        self.num_features = num_features # Features per node
        self.hidden_dim = hidden_dim
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.dropout = dropout
        self.num_nodes = len(node_mapping) # Explicitly get the number of nodes

        # SPATIAL: Graph Convolution Layers
        self.gcn_layers = nn.ModuleList()
        self.gcn_layers.append(GCNConv(num_features, hidden_dim))

        for _ in range(num_layers - 1):
            self.gcn_layers.append(GCNConv(hidden_dim, hidden_dim))

        # TEMPORAL: LSTM for time series
        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            dropout=dropout,
            bidirectional=False
        )

        # ATTENTION: Temporal attention mechanism
        self.temporal_attention = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=4,
            dropout=dropout,
            batch_first=True
        )

        # PREDICTION: Output layers
        self.prediction_layers = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.ReLU(),
            nn.Linear(hidden_dim // 4, pred_len)  # Predict multiple time steps
        )

        # Batch normalization
        self.batch_norm = nn.BatchNorm1d(hidden_dim)

        print(f"✅ GNN Model Initialized (Class Definition Only).")
        print(f"   • Input features (per node): {num_features}")
        print(f"   • Number of nodes: {self.num_nodes}")
        print(f"   • Hidden dim: {hidden_dim}")
        print(f"   • GCN layers: {num_layers}")
        print(f"   • Sequence length: {seq_len}")
        print(f"   • Prediction horizon: {pred_len}")

    def forward(self, x, edge_index, batch=None):
        """
        Forward pass
        x: [batch_size, seq_len, num_nodes, num_features] (expected input)
        edge_index: [2, num_edges]
        """
        # DEBUG: Print the shape of x as it enters the forward method
        print(f"DEBUG: x.shape inside forward (raw input): {x.shape}")

        # Unpack dimensions assuming x is already [batch_size, seq_len, num_nodes, num_features]
        batch_size, seq_len, num_nodes, num_features = x.shape

        # Process each time step through GNN
        spatial_outputs = []

        for t in range(seq_len):
            # Get features for time step t: [batch_size * num_nodes, num_features]
            x_t = x[:, t, :, :].reshape(-1, num_features)

            # Apply GCN layers
            h = x_t
            for gcn_layer in self.gcn_layers:
                h = gcn_layer(h, edge_index)
                h = F.relu(h)
                h = F.dropout(h, p=self.dropout, training=self.training)

            spatial_outputs.append(h)

        # Stack temporal dimension: [batch_size * num_nodes, seq_len, hidden_dim]
        spatial_temporal = torch.stack(spatial_outputs, dim=1)

        # Reshape for LSTM: [batch_size, num_nodes, seq_len, hidden_dim] -> [batch_size * num_nodes, seq_len, hidden_dim]
        lstm_input = spatial_temporal.reshape(batch_size * num_nodes, seq_len, self.hidden_dim)
        lstm_out, (hidden, cell) = self.lstm(lstm_input)

        # Apply temporal attention
        attended_out, attention_weights = self.temporal_attention(
            lstm_out, lstm_out, lstm_out
        )

        # Take the last time step's output
        last_hidden = attended_out[:, -1, :]  # [batch_size * num_nodes, hidden_dim]

        # Batch normalization
        last_hidden = self.batch_norm(last_hidden)

        # Make predictions
        predictions = self.prediction_layers(last_hidden)

        # Reshape back: [batch_size, num_nodes, pred_len]
        predictions = predictions.reshape(batch_size, num_nodes, self.pred_len)

        return predictions, attention_weights


#STEP 2: DATA LOADER & TRAINING SETUP

In [ ]:
def create_data_loaders(dataset, batch_size=32, train_ratio=0.8):
    """Create train and validation data loaders"""
    # Calculate split sizes
    total_size = len(dataset)
    train_size = int(train_ratio * total_size)
    val_size = total_size - train_size

    # Split dataset
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    print(f"📦 Data Loaders Created:")
    print(f"   • Training samples: {len(train_dataset)}")
    print(f"   • Validation samples: {len(val_dataset)}")
    print(f"   • Batch size: {batch_size}")

    return train_loader, val_loader

# Create data loaders
print("\n📦 Creating data loaders...")
train_loader, val_loader = create_data_loaders(dataset, batch_size=16)

# Setup training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Using device: {device}")

# Initialize the model
print("🤖 Initializing GNN Model...")
model = SpatioTemporalGNN(
    num_features=len(feature_cols),  # From previous step
    hidden_dim=64,
    num_layers=2,
    seq_len=30,
    pred_len=7,
    dropout=0.2
)

# Print model summary
print(f"\n📊 Model Architecture:")
print(model)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

# Debug check BEFORE moving model to device
if hasattr(model, 'gcn_layers') and len(model.gcn_layers) > 0:
    print(f"DEBUG: GCN layer 0 weight device BEFORE .to(device): {model.gcn_layers[0].lin.weight.device}")

# Move model to device
model = model.to(device)
edge_index = edge_index.to(device)

# Debug check AFTER moving model to device
if hasattr(model, 'gcn_layers') and len(model.gcn_layers) > 0:
    print(f"DEBUG: GCN layer 0 weight device AFTER .to(device): {model.gcn_layers[0].lin.weight.device}")

# Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

print(f"⚙️  Training Setup:")
print(f"   • Loss function: MSE")
print(f"   • Optimizer: Adam")
print(f"   • Learning rate: 0.001")
print(f"   • Weight decay: 1e-5")


#🏋️ STEP 3: TRAINING LOOP

In [ ]:
def train_model(model, train_loader, val_loader, epochs=100):
    """Training loop with validation"""
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    patience = 10
    patience_counter = 0

    print(f"\n🎯 Starting Training for {epochs} epochs...")
    print("=" * 60)

    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0

        for batch_idx, (sequences, targets) in enumerate(train_loader):
            sequences = sequences.to(device)  # [batch, seq_len, num_nodes, features]
            targets = targets.to(device)      # [batch, num_nodes, pred_len]

            optimizer.zero_grad()

            # Forward pass
            predictions, attention_weights = model(sequences, edge_index)

            # Calculate loss
            loss = criterion(predictions, targets)

            # Backward pass
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            if batch_idx % 10 == 0:
                print(f"   Epoch {epoch+1:03d} | Batch {batch_idx:03d} | Loss: {loss.item():.6f}")

        # Validation phase
        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for sequences, targets in val_loader:
                sequences = sequences.to(device)
                targets = targets.to(device)

                predictions, _ = model(sequences, edge_index)
                loss = criterion(predictions, targets)
                val_loss += loss.item()

        # Calculate average losses
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)

        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)

        # Learning rate scheduling
        scheduler.step(avg_val_loss)
        current_lr = optimizer.param_groups[0]['lr']

        # Print progress
        print(f"📈 Epoch {epoch+1:03d}/{epochs} | "
              f"Train Loss: {avg_train_loss:.6f} | "
              f"Val Loss: {avg_val_loss:.6f} | "
              f"LR: {current_lr:.6f}")

        # Early stopping check
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0

            # Save best model
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': avg_train_loss,
                'val_loss': avg_val_loss,
            }, COLAB_PROJECT_ROOT / 'models' / 'best_gnn_model.pth')

            print(f"💾 New best model saved! Val Loss: {best_val_loss:.6f}")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"🛑 Early stopping triggered after {epoch+1} epochs")
                break

    print(f"\n✅ Training completed!")
    print(f"🎯 Best validation loss: {best_val_loss:.6f}")

    return train_losses, val_losses

# Create models directory
models_path = COLAB_PROJECT_ROOT / 'models'
models_path.mkdir(parents=True, exist_ok=True)

# Start training!
print("🚀 Starting GNN training...")
train_losses, val_losses = train_model(model, train_loader, val_loader, epochs=100)

#STEP 4: TRAINING VISUALIZATION

In [ ]:
def plot_training_progress(train_losses, val_losses):
    """Plot training and validation losses"""
    import matplotlib.pyplot as plt

    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Training Loss', linewidth=2)
    plt.plot(val_losses, label='Validation Loss', linewidth=2)
    plt.title('GNN Training Progress', fontsize=14, fontweight='bold')
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.yscale('log')  # Log scale for better visualization
    plt.tight_layout()
    plt.show()

    # Print final metrics
    print(f"📊 Final Training Metrics:")
    print(f"   • Final train loss: {train_losses[-1]:.6f}")
    print(f"   • Final val loss: {val_losses[-1]:.6f}")
    print(f"   • Improvement: {((train_losses[0] - train_losses[-1]) / train_losses[0] * 100):.1f}%")

# Plot training progress
plot_training_progress(train_losses, val_losses)

#Model Evaluation

In [ ]:
def evaluate_model(model, val_loader, scalers, node_mapping): # Changed parameter name
    """Comprehensive model evaluation"""
    model.eval()
    all_predictions = []
    all_targets = []

    print("\n🔍 Model Evaluation...")

    with torch.no_grad():
        for sequences, targets in val_loader:
            sequences = sequences.to(device)
            targets = targets.to(device)

            predictions, attention_weights = model(sequences, edge_index)

            all_predictions.append(predictions.cpu().numpy())
            all_targets.append(targets.cpu().numpy())

    # Concatenate all batches
    all_predictions = np.concatenate(all_predictions, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)

    # Convert back to original scale
    water_scaler = scalers['adhala']  # Use Adhala's scaler as reference

    # Calculate metrics
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

    mse = mean_squared_error(all_targets.flatten(), all_predictions.flatten())
    mae = mean_absolute_error(all_targets.flatten(), all_predictions.flatten())
    rmse = np.sqrt(mse)
    r2 = r2_score(all_targets.flatten(), all_predictions.flatten())

    print(f"📊 Evaluation Metrics:")
    print(f"   • MSE: {mse:.6f}")
    print(f"   • MAE: {mae:.6f}")
    print(f"   • RMSE: {rmse:.6f}")
    print(f"   • R² Score: {r2:.4f}")

    # Per-node metrics
    print(f"\n🌊 Per-Lake Metrics:")
    for node_idx, lake_name in node_mapping.items():
        node_mse = mean_squared_error(
            all_targets[:, node_idx, :].flatten(),
            all_predictions[:, node_idx, :].flatten()
        )
        node_rmse = np.sqrt(node_mse)
        print(f"   • {lake_name.title()}: RMSE = {node_rmse:.4f}")

    return all_predictions, all_targets

# Evaluate the model
print("📊 Evaluating model performance...")
predictions, targets = evaluate_model(model, val_loader, scalers, node_mapping)


#STEP 6: PREDICTION VISUALIZATION

In [ ]:
def plot_predictions(predictions, targets, node_mapping, num_samples=5):
    """Visualize model predictions vs actual values"""
    import matplotlib.pyplot as plt

    num_nodes = len(node_mapping)
    fig, axes = plt.subplots(num_nodes, 1, figsize=(12, 4 * num_nodes))

    if num_nodes == 1:
        axes = [axes]

    for node_idx, (lake_name, ax) in enumerate(zip(node_mapping.values(), axes)):
        # Plot a few samples
        for sample_idx in range(min(num_samples, predictions.shape[0])):
            pred = predictions[sample_idx, node_idx, :]
            actual = targets[sample_idx, node_idx, :]

            time_steps = range(len(pred))
            ax.plot(time_steps, pred, alpha=0.7, linewidth=1,
                   label=f'Predicted' if sample_idx == 0 else "")
            ax.plot(time_steps, actual, alpha=0.7, linewidth=1,
                   label=f'Actual' if sample_idx == 0 else "")

        ax.set_title(f'{lake_name.title()} - Predictions vs Actual', fontweight='bold')
        ax.set_xlabel('Time Steps (Days)')
        ax.set_ylabel('Water Level (Normalized)')
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# Visualize predictions
print("\n🎨 Visualizing predictions...")
plot_predictions(predictions, targets, node_mapping)

#STEP 7: SAVE COMPLETE MODEL

In [ ]:
def save_complete_model(model, train_losses, val_losses, feature_cols, node_mapping):
    """Save the complete trained model and training history"""
    model_path = COLAB_PROJECT_ROOT / 'models'

    # Save final model
    final_model_path = model_path / 'final_gnn_model.pth'
    torch.save({
        'model_state_dict': model.state_dict(),
        'feature_columns': feature_cols,
        'node_mapping': node_mapping,
        'training_losses': train_losses,
        'validation_losses': val_losses,
        'model_architecture': model.__class__.__name__,
        'timestamp': pd.Timestamp.now()
    }, final_model_path)

    # Save training history
    history_df = pd.DataFrame({
        'epoch': range(1, len(train_losses) + 1),
        'train_loss': train_losses,
        'val_loss': val_losses
    })
    history_path = model_path / 'training_history.csv'
    history_df.to_csv(history_path, index=False)

    print(f"💾 Complete model saved to: {final_model_path}")
    print(f"📊 Training history saved to: {history_path}")
    print(f"🎉 GNN training pipeline completed successfully!")

# Save everything
save_complete_model(model, train_losses, val_losses, feature_cols, node_mapping)